# Waymo dataset coverage check

Cross-check `train/metadata_vace_general.csv` against the videos actually present on disk in `/miele/liory/datasets/waymo/videos/` (the canonical copy — the `~/ARV2V/waymo/` directory is only a sparse local subset).

The metadata describes the full preprocessed dataset (798 scenes × 5 cameras × ~15 sliding windows = 61,520 clips). Downstream training only needs the **forward camera (`camera_id=0`)**, so this notebook focuses on confirming that every `camera_id=0` row has a matching mp4 on disk.

Logic lives in the companion script `check_dataset.py` so the notebook stays thin.

In [17]:
from pathlib import Path

import pandas as pd

WAYMO_ROOT = Path('/miele/liory/datasets/waymo')
META = WAYMO_ROOT / 'train' / 'metadata_vace_general.csv'
VIDEOS_DIR = WAYMO_ROOT / 'videos'

print('metadata:', META)
print('videos:  ', VIDEOS_DIR)

## Load metadata into a DataFrame

The CSV is loaded as `df`. We add two derived columns: `video_filename` (basename of the `video` path) and `file_present` (boolean: does the referenced clip exist in `videos/`?).

In [ ]:
df = pd.read_csv(META)
df['video_filename'] = df['video'].str.rsplit('/', n=1).str[-1]

present = {p.name for p in VIDEOS_DIR.iterdir() if p.suffix == '.mp4'}
df['file_present'] = df['video_filename'].isin(present)

print(f'rows: {len(df):,}   columns: {len(df.columns)}')
print(f'with file on disk: {df["file_present"].sum():,}   missing: {(~df["file_present"]).sum():,}')
df[12302:12310]

In [19]:
df.dtypes

In [20]:
# coverage broken down by camera_id
coverage = (
    df.groupby('camera_id')
      .agg(meta_rows=('video', 'size'),
           with_file=('file_present', 'sum'),
           scenes_total=('scene_id', 'nunique'),
           scenes_with_file=('scene_id', lambda s: s[df.loc[s.index, 'file_present']].nunique()))
)
coverage['coverage_pct'] = (100 * coverage['with_file'] / coverage['meta_rows']).round(2)
coverage

In [21]:
# camera_id=0 (forward) rows — the subset we actually need for training.
front = df.query('camera_id == 0').copy()
print(f'camera_id=0 rows: {len(front):,}')
print(f'with file on disk: {front["file_present"].sum():,}   missing: {(~front["file_present"]).sum():,}')
print(f'unique scenes:     {front["scene_id"].nunique():,}')
front.head()

### Interpretation

Confirmed against `/miele/liory/datasets/waymo/` (read-only — nothing copied locally):

- **Full coverage**: all 61,520 metadata rows have a matching mp4 on disk (100% across every camera).
- **camera_id=0 (forward) — the only subset we need for training**: **12,304 / 12,304** rows present, spanning all **798** scenes.
- Other cameras (1–4) are also fully present, but unused downstream — leave them where they are.

Conclusion: training can read `camera_id=0` clips directly from `/miele/liory/datasets/waymo/videos/` without any local mirroring.

In [22]:
# camera_id=0 file paths on disk — ready to feed to a dataloader
front_paths = [str(VIDEOS_DIR / f) for f in df.query('camera_id == 0')['video_filename']]
print(f'{len(front_paths):,} forward-camera clips')
print('first 3:', front_paths[:3])